# log-samples-eval-callback composite — cx20: increment step counter, then fire log-samples callback every N

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `log-samples-eval-callback`, `step-counter-increment`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader, TensorDataset

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "log-samples-eval-callback"
DD_ATOM_IDS = ["log-samples-eval-callback", "step-counter-increment"]
DD_SUBTOPICS = ["Logging: log-samples eval callback", "Trainer: step counter increment"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A trainer that logs generated samples every `N` steps depends on two atoms wired in exactly the right order:

1. **step-counter-increment** — increment `self.step` (or the local `step` var) AT THE TOP of each iteration, so the FIRST iteration ends with `step == 1`, not `step == 0`. The convention `step % N == 0` then fires correctly on iteration `N`, not iteration 0.
2. **log-samples-eval-callback** — when `step % N == 0`, call `sample_fn()` and stash the result in a sink keyed by step.

**Why order matters.** If you check `step % N == 0` BEFORE incrementing, step 0 (before any work was done) fires the callback — you log random-init samples. If you check AFTER incrementing, the callback fires on steps `N, 2N, 3N, ...` — sample sets that reflect actual training progress.

**Anatomy.**
```python
for micro_batch in batches:
    step += 1                          # step-counter-increment (top of loop).
    # ... train ...
    if step % log_every == 0:          # log-samples-eval-callback.
        samples = sample_fn()
        sink[step] = samples
```

### Composite Exercise — increment step counter, then fire log-samples callback every N

**Atoms exercised together**: `log-samples-eval-callback`, `step-counter-increment`

Implement `cx20_train_with_sample_logging(n_iters, log_every, sample_fn)`.

Inputs:
- `n_iters` — int. Number of training iterations to run.
- `log_every` — int. Sample callback fires when `step % log_every == 0`.
- `sample_fn` — callable `(step: int) -> Any`. Receives the current step number and returns a sample.

Required behaviour:
1. Initialise a local `step = 0` counter.
2. Initialise a `sink` dict: `{step: sample}` entries appended by the callback.
3. For each of `n_iters` iterations:
   a. Increment `step` (atom: step-counter-increment) — so iteration 1 ends with step=1.
   b. If `step % log_every == 0` (atom: log-samples-eval-callback), call `sample_fn(step)` and write `sink[step] = sample`.
4. Return `(step, sink)` — the final step counter and the populated sink.

Test checks:
- Step counter increments exactly `n_iters` times (final step == n_iters).
- Callback fires on steps `log_every, 2*log_every, ..., (n_iters // log_every) * log_every`.
- `sink` keys are exactly that arithmetic progression.
- `sample_fn` is called with the correct `step` argument each time.
- Step 0 does NOT fire the callback (because counter starts at 0 and increments BEFORE the check).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx20_train_with_sample_logging(n_iters, log_every, sample_fn):
    """Run n_iters of a fake training loop; log samples every log_every steps. Returns (final_step, sink_dict)."""
    raise NotImplementedError

def _test_cx20():
    # Case A: 10 iters, log every 3 → fires on steps 3, 6, 9.
    received_steps = []
    def _sample_fn(step):
        received_steps.append(step)
        return f'samples@{step}'
    final_step, sink = cx20_train_with_sample_logging(n_iters=10, log_every=3, sample_fn=_sample_fn)
    assert final_step == 10, f'final step should equal n_iters=10; got {final_step}'
    assert sorted(sink.keys()) == [3, 6, 9], f'expected keys [3, 6, 9]; got {sorted(sink.keys())}'
    assert sink[3] == 'samples@3'
    assert sink[6] == 'samples@6'
    assert sink[9] == 'samples@9'
    assert received_steps == [3, 6, 9], f'sample_fn called with wrong steps; got {received_steps}'

    # Case B: zero iters → no callback, final step 0.
    calls = []
    fs, sk = cx20_train_with_sample_logging(n_iters=0, log_every=2, sample_fn=lambda s: calls.append(s) or 'x')
    assert fs == 0
    assert sk == {}
    assert calls == [], 'no iterations should mean no callback'

    # Case C: log_every=1 fires every step.
    fired = []
    fs2, sk2 = cx20_train_with_sample_logging(
        n_iters=5, log_every=1, sample_fn=lambda s: fired.append(s) or s
    )
    assert fs2 == 5
    assert sorted(sk2.keys()) == [1, 2, 3, 4, 5], (
        f'log_every=1 should fire on every step; got keys {sorted(sk2.keys())}'
    )
    assert fired == [1, 2, 3, 4, 5]

    # Case D: log_every > n_iters → no callback fires.
    fs3, sk3 = cx20_train_with_sample_logging(n_iters=4, log_every=10, sample_fn=lambda s: 'x')
    assert fs3 == 4
    assert sk3 == {}, 'log_every > n_iters should produce no callbacks'

    # Case E: counter increments BEFORE the modulo check — so step 0 NEVER fires.
    # If a buggy impl checked `step % log_every == 0` BEFORE incrementing, step=0 would fire.
    fired_b = []
    fs4, sk4 = cx20_train_with_sample_logging(
        n_iters=1, log_every=1, sample_fn=lambda s: fired_b.append(s) or s
    )
    # Right answer: fires once on step 1; sink == {1: 1}.
    # Wrong answer (check-before-increment): fires twice — step 0 and step 1.
    assert fired_b == [1], f'should fire exactly once on step 1; got {fired_b}'
    assert sk4 == {1: 1}, f'sink should be {{1: 1}}; got {sk4}'
    _dd_passed.add('cx20')

_test_cx20()

<details><summary>Show solution — cx20</summary>

```python
def cx20_train_with_sample_logging(n_iters, log_every, sample_fn):
    step = 0
    sink = {}
    for _ in range(n_iters):
        # Atom A (step-counter-increment): bump BEFORE the modulo check so step 0 never fires.
        step += 1
        # Atom B (log-samples-eval-callback): every log_every steps, call sample_fn.
        if step % log_every == 0:
            sink[step] = sample_fn(step)
    return step, sink
```

The increment-then-check pattern matches PyTorch's `Optimizer.step()` convention (step counter incremented inside `.step()` before any logging hooks observe it). The alternative — check-then-increment — leads to off-by-one bugs where you log untrained model output at step 0, then log every `log_every` thereafter.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx20'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx20',
        'subtopics': ["Logging: log-samples eval callback", "Trainer: step counter increment"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()